# Lab 3, Notebook A — The Gaussian, computed

*Linear Algebra for Data Science, UCU 2026. Uses Session 4.1 (spectral theorem, Cholesky, matrix square root) and Session 4.2 (covariance, whitening, Mahalanobis distance, the Gaussian density).*

The multivariate normal density needs $\Sigma^{-1}$ and $\det\Sigma$, whitening needs a square root of $\Sigma$, and sampling needs any factor $B$ with $BB^\top = \Sigma$. In this notebook you will compute all of them from **one Cholesky factorization** $\Sigma = LL^\top$, never forming an inverse or a determinant. You will also see each of them fail when the matrix is nearly singular.

**How to work.** Most tasks ask you to **predict first, in writing, then compute, then explain any difference**. Write the prediction *before* running the cell. A wrong prediction with an honest explanation earns more at the defence than a correct number you cannot explain.

**Rules.**
- Inside the functions you implement, do not call `np.linalg.inv`, `np.linalg.det` or `scipy.stats.multivariate_normal`. Use them only where a cell says **check** (and in `gauss_logpdf_naive`, which exists to be compared).
- You may use `scipy.linalg.cholesky`, `solve_triangular`, `cho_factor`/`cho_solve` and `np.linalg.eigh`.
- Do not change the seeds.
- Before you submit, run `Kernel → Restart & Run All`.

**Marking.** Core A is worth 2 points at the defence; the code point and the extension are shared with Notebook B.

**Contents.** A1 Covariance and whitening · A2 `log det` without `det` · A3 Sampling · Extensions E1 (repairing a correlation matrix) and E2 (Fisher's discriminant). Attempt **at most one** extension in the whole lab.

| | |
|---|---|
| **Team member 1** | *name, surname* |
| **Team member 2** | *name, surname* |
| **Extension attempted** (at most one in the whole lab) | *none / E1 / E2 / E3* |
| **Late days used on this lab** | *0* |
| **Runs top to bottom from a clean kernel** | *yes / no* |

In [2]:
import sys, time
import numpy as np
import scipy, scipy.linalg as sl
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer, load_wine

print("python", sys.version.split()[0], "| numpy", np.__version__, "| scipy", scipy.__version__,
      "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__, "| sklearn", sklearn.__version__)
np.set_printoptions(precision=4, suppress=False, linewidth=110)

python 3.14.6 | numpy 2.5.3 | scipy 1.18.1 | pandas 3.0.6 | matplotlib 3.11.2 | sklearn 1.9.1


---
## A1. Covariance and whitening

The data set has 569 breast tumours and 30 features of the cell nuclei (radius, texture, area, smoothness, ...), measured in very different units. Rows are observations, columns are features.

In [5]:
data = load_breast_cancer()
X = data.data
n, p = X.shape
print(n, "tumours,", p, "features; first five:", list(data.feature_names[:5]))

569 tumours, 30 features; first five: [np.str_('mean radius'), np.str_('mean texture'), np.str_('mean perimeter'), np.str_('mean area'), np.str_('mean smoothness')]


**(a)** Implement `sample_cov(X)`, the sample covariance $\operatorname{Cov} = \frac1n X_c^\top X_c$ of the centred data. The course uses the divisor $1/n$, so do not use `np.cov`, which divides by $n-1$. Then implement `sample_corr(X)`, the sample correlation matrix.

In [6]:
def sample_cov(X):
    # YOUR CODE HERE
    raise NotImplementedError

def sample_corr(X):
    # YOUR CODE HERE
    raise NotImplementedError

In [7]:
# check
C, R = sample_cov(X), sample_corr(X)
assert np.allclose(C, np.cov(X, rowvar=False, bias=True))
assert np.allclose(R, np.corrcoef(X, rowvar=False))
for name, M in [("Cov", C), ("Corr", R)]:
    ev = np.linalg.eigvalsh(M)
    print(f"{name}: symmetric {np.allclose(M, M.T)}, eigenvalues in [{ev[0]:.3e}, {ev[-1]:.3e}]")

NotImplementedError: 

**(b)** *Predict, then compute.* Which of `Cov` and `Corr` do you expect to be worse conditioned, and why? Write your prediction, then compute both condition numbers from the eigenvalues. What does the difference tell you about the units of the features?

**Your prediction** *(write it before running the next cell)*:

In [ ]:
def cond_sym(M):
    ev = np.linalg.eigvalsh(M)
    return ev[-1] / ev[0]

# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(c)** Implement three whitening maps. Each takes the data `X` and returns the whitened centred data `W` ($n\times p$), whose sample covariance should be the identity:

| function | map | factorization |
|---|---|---|
| `whiten_chol(X)` | $A = L^{-1}$ | $\operatorname{Cov} = LL^\top$ |
| `whiten_pca(X)` | $A = D^{-1/2}U^\top$ | $\operatorname{Cov} = UDU^\top$ |
| `whiten_zca(X)` | $A = \operatorname{Cov}^{-1/2}$ | symmetric square root |

Apply the map with solves and products, never with an inverse. Rows of `W` are $A(\mathbf{x}_i - \bar{\mathbf{x}})$.

In [ ]:
def whiten_chol(X):
    # YOUR CODE HERE
    raise NotImplementedError

def whiten_pca(X):
    # YOUR CODE HERE
    raise NotImplementedError

def whiten_zca(X):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# check: the sample covariance of each whitened data set is I, to rounding error
whiteners = {"Cholesky": whiten_chol, "PCA": whiten_pca, "ZCA": whiten_zca}
for name, f in whiteners.items():
    W = f(X)
    err = np.abs(sample_cov(W) - np.eye(p)).max()
    print(f"{name:8s} max |Cov(W) - I| = {err:.1e}")
    assert err < 1e-6

*Question.* The three errors above are all small, but they differ by several orders of magnitude. Which map is most accurate here, and why might the two that use `eigh` lose more digits on this matrix? (Look back at (b).)

**Your answer.** *Write it here.*

**(d)** For each whitening compute the mean squared displacement $\frac1n\sum_i \|\mathbf{w}_i - (\mathbf{x}_i-\bar{\mathbf{x}})\|^2$. Which is smallest? State the theorem from Session 4.2 that predicts this.

The data are in very different units, so the displacement is dominated by the large-scale features. Repeat on the **standardised** data (each column centred and divided by its standard deviation). Does the ranking change? Do the differences become easier to see?

In [ ]:
def msd(W, X):
    # YOUR CODE HERE
    raise NotImplementedError

Xs = (X - X.mean(axis=0)) / X.std(axis=0)      # standardised data, used again below
for label, Z in [("raw", X), ("standardised", Xs)]:
    print(label, {name: round(float(msd(f(Z), Z)), 3) for name, f in whiteners.items()})

**Your answer.** *Write it here.*

**(e)** The **Mahalanobis distance** of $\mathbf{x}$ from the mean is $d(\mathbf{x}) = \sqrt{(\mathbf{x}-\bar{\mathbf{x}})^\top\operatorname{Cov}^{-1}(\mathbf{x}-\bar{\mathbf{x}})}$.

1. Compute it for every tumour in three ways: as the Euclidean norm of the rows of each of the three whitened data sets from (c). Show that the three agree to rounding error, and explain in two sentences why they must, using $A^\top A$.
2. Implement `mahalanobis(x, mu, L)` for **one** new point, given the mean and the Cholesky factor $L$ of `Cov`. Use one triangular solve, so the cost is $O(p^2)$ per point. Check it against part 1 on all tumours, and list the five tumours with the largest distance.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

In [ ]:
def mahalanobis(x, mu, L):
    # YOUR CODE HERE
    raise NotImplementedError

mu, L = X.mean(axis=0), sl.cholesky(sample_cov(X), lower=True)
d_one = np.array([mahalanobis(x, mu, L) for x in X])
# check
assert np.allclose(d_one, np.linalg.norm(whiten_chol(X), axis=1))
top5 = np.argsort(d_one)[::-1][:5]
print("five most outlying tumours:", top5, "distances", np.round(d_one[top5], 2))

**(f)** *When $p$ approaches $n$.* Draw random subsamples of $m$ rows for $m = 500, 100, 50, 35, 31, 30, 25$, and for each compute the smallest eigenvalue of the subsample's correlation matrix.

*Predict before running:* at which $m$ does this eigenvalue become zero, up to rounding? Justify your prediction from the rank of the centred data.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
rng = np.random.default_rng(0)
for m in [500, 100, 50, 35, 31, 30, 25]:
    idx = rng.choice(n, m, replace=False)
    # YOUR CODE HERE
    raise NotImplementedError
    print(f"m = {m:3d}: smallest eigenvalue of Corr = {lam_min: .2e}")

**Your answer.** *Write it here.*

Now take **one** subsample of $m = 35$ rows as a "training set" and treat the other 534 rows as new data from the same distribution. Standardise both with the training means and standard deviations, and build the ZCA map from the training covariance $S$. Then repeat with **shrinkage**, $S + \alpha I$, for $\alpha = 0, 10^{-3}, 10^{-2}, 10^{-1}, 1$.

Implement `zca_map(S, alpha)`, which returns $(S+\alpha I)^{-1/2}$. For each $\alpha$, report:
- the largest amplification factor $1/\sqrt{\lambda_{\min}(S)+\alpha}$;
- the largest and smallest eigenvalue of the covariance of the **new** data after whitening. Ideally both are 1.

Describe the trade-off in a short paragraph.

In [ ]:
def zca_map(S, alpha):
    # YOUR CODE HERE
    raise NotImplementedError

rng = np.random.default_rng(1)
perm = rng.permutation(n)
tr, te = perm[:35], perm[35:]
m_tr, s_tr = X[tr].mean(axis=0), X[tr].std(axis=0)
A_tr, A_te = (X[tr] - m_tr) / s_tr, (X[te] - m_tr) / s_tr
S = A_tr.T @ A_tr / len(tr)
print(f"smallest eigenvalue of S: {np.linalg.eigvalsh(S)[0]:.2e}")
for alpha in [0, 1e-3, 1e-2, 1e-1, 1]:
    T = zca_map(S, alpha)
    ev = np.linalg.eigvalsh(sample_cov(A_te @ T))
    amp = 1 / np.sqrt(np.linalg.eigvalsh(S)[0] + alpha)
    print(f"alpha = {alpha:6g}: amplification {amp:8.1f};  new data after whitening: "
          f"eigenvalues of Cov in [{ev[0]:.3f}, {ev[-1]:.1f}]")

**Your answer.** *Write it here.*

---
## A2. `log det` without `det`

The log-density of $\mathcal{N}(\boldsymbol\mu,\Sigma)$ in $\mathbb{R}^p$ is

$$\log p(\mathbf{x}) = -\tfrac p2\log(2\pi) - \tfrac12\log\det\Sigma - \tfrac12(\mathbf{x}-\boldsymbol\mu)^\top\Sigma^{-1}(\mathbf{x}-\boldsymbol\mu).$$

With $\Sigma = LL^\top$: $\log\det\Sigma = 2\sum_j\log l_{jj}$, and the quadratic form is $\|\mathbf{w}\|^2$ where $L\mathbf{w} = \mathbf{x}-\boldsymbol\mu$.

**(a)** Implement `gauss_logpdf(x, mu, Sigma)` for one point, using one Cholesky factorization and one triangular solve. **Check** it against `scipy.stats.multivariate_normal.logpdf` on the breast cancer correlation matrix.

**(b)** Implement the naive version `gauss_logpdf_naive`, which uses `np.linalg.det` and `np.linalg.inv`. This is the only place they are allowed.

In [ ]:
def gauss_logpdf(x, mu, Sigma):
    # YOUR CODE HERE
    raise NotImplementedError

def gauss_logpdf_naive(x, mu, Sigma):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# check
from scipy.stats import multivariate_normal
x0, zero = Xs[0], np.zeros(p)
ref = multivariate_normal(mean=zero, cov=R).logpdf(x0)
print("Cholesky:", gauss_logpdf(x0, zero, R), " naive:", gauss_logpdf_naive(x0, zero, R), " scipy:", ref)
assert np.isclose(gauss_logpdf(x0, zero, R), ref)

**(c)** *Predict, then compute.* Take the AR(1) covariance $\Sigma_{ij} = \rho^{|i-j|}$ with $\rho = 0.99$. It describes a strongly autocorrelated time series and has a closed-form determinant, $\det\Sigma = (1-\rho^2)^{p-1}$.
- Use this formula to predict the dimension $p$ at which `np.linalg.det` stops returning an accurate value, and the dimension at which it returns exactly zero. Doubles below about $2.2\cdot10^{-308}$ are *subnormal* and carry fewer digits; the smallest positive double is about $4.9\cdot10^{-324}$.
- Then sweep $p$ from 150 to 220. Plot the error of $\log\det\Sigma$ computed both ways against the exact value $(p-1)\log(1-\rho^2)$. Report where the naive one first goes wrong, and how.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
def ar1(p, rho):
    i = np.arange(p)
    return rho ** np.abs(i[:, None] - i[None, :])

rho = 0.99
ps = np.arange(150, 221)
err_chol, err_naive = [], []
for q in ps:
    S = ar1(q, rho)
    exact = (q - 1) * np.log(1 - rho ** 2)
    # YOUR CODE HERE
    raise NotImplementedError
    err_chol.append(abs(ld_chol - exact))
    err_naive.append(abs(ld_naive - exact))

err_naive = np.array(err_naive)
first_bad = ps[np.argmax(err_naive > 1e-8)]
first_inf = ps[np.argmax(~np.isfinite(err_naive))]
print("naive log det first off by > 1e-8 at p =", first_bad, "; first infinite at p =", first_inf)
plt.figure(figsize=(6, 3.5))
plt.semilogy(ps, err_chol, "o-", ms=3, label="Cholesky")
plt.semilogy(ps, np.where(np.isfinite(err_naive), err_naive, np.nan), "s-", ms=3, label="np.linalg.det")
plt.axvline(first_inf, color="grey", ls=":", label="det = 0 from here")
plt.xlabel("dimension p"); plt.ylabel("|error in log det|"); plt.legend(); plt.tight_layout(); plt.show()

**Your answer.** *Write it here.*

**(d)** Time both implementations of the log-density for $p = 200, 400, 800, 1600$, on the AR(1) matrix or any symmetric positive definite matrix you like. Explain the ratio from the flop counts: Cholesky costs about $p^3/3$, LU (inside `det`) about $2p^3/3$, and an explicit inverse about $2p^3$.

In [ ]:
def best_time(f, *args, repeat=3):
    best = np.inf
    for _ in range(repeat):
        t0 = time.perf_counter(); f(*args); best = min(best, time.perf_counter() - t0)
    return best

for q in [200, 400, 800, 1600]:
    S = ar1(q, 0.5)
    x, m0 = np.ones(q), np.zeros(q)
    t_c, t_n = best_time(gauss_logpdf, x, m0, S), best_time(gauss_logpdf_naive, x, m0, S)
    print(f"p = {q:5d}: Cholesky {1e3*t_c:8.2f} ms, naive {1e3*t_n:8.2f} ms, ratio {t_n/t_c:4.1f}")

**Your answer.** *Write it here.*

---
## A3. Sampling

If $\mathbf{z}\sim\mathcal{N}(0,I)$ and $BB^\top=\Sigma$, then $\boldsymbol\mu + B\mathbf{z}\sim\mathcal{N}(\boldsymbol\mu,\Sigma)$, for **any** such $B$. The Cholesky factor is the cheapest choice.

**(a)** Implement `gauss_sample(mu, Sigma, N, rng)`, returning an $N\times p$ array of samples $\boldsymbol\mu + L\mathbf{z}$.

In [ ]:
def gauss_sample(mu, Sigma, N, rng):
    # YOUR CODE HERE
    raise NotImplementedError

**(b)** *Predict, then compute.* How should the error $\|\operatorname{Cov}_N - \Sigma\|_F$ of the sample covariance of $N$ samples shrink as $N$ grows? Measure it for $\Sigma$ = the breast cancer correlation matrix and $N = 10^2,\dots,10^5$. Plot it on log–log axes and fit the slope.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
rng = np.random.default_rng(2)
Ns = np.array([10 ** k for k in [2, 2.5, 3, 3.5, 4, 4.5, 5]]).astype(int)
errs = []
for N in Ns:
    Y = gauss_sample(np.zeros(p), R, N, rng)
    errs.append(np.linalg.norm(sample_cov(Y) - R))
slope = np.polyfit(np.log(Ns), np.log(errs), 1)[0]
print(f"fitted slope: {slope:.3f}")
plt.figure(figsize=(5, 3.5)); plt.loglog(Ns, errs, "o-")
plt.xlabel("N"); plt.ylabel(r"$\|Cov_N - \Sigma\|_F$"); plt.tight_layout(); plt.show()

**Your answer.** *Write it here.*

**(c)** Take the covariance of a random subsample of **20** rows of the standardised data `Xs` (seed below). It is a singular PSD matrix: why?
1. Show that `cholesky` fails on it.
2. Write `gauss_sample_psd(mu, Sigma, N, rng)`, which uses a factor from `eigh` instead: any $B$ with $BB^\top=\Sigma$. Set every eigenvalue below $10^{-12}\lambda_{\max}$ to zero. Why is it not enough to clip only the negative ones? (Hint: what is $\sqrt{10^{-16}}$?)
3. *Before checking, predict:* in which subspace do the samples lie? Verify by projecting the samples onto the (numerical) null space of $\Sigma$.

In [ ]:
rng = np.random.default_rng(3)
S20 = sample_cov(Xs[rng.choice(n, 20, replace=False)])
try:
    sl.cholesky(S20, lower=True)
    print("Cholesky succeeded")
except np.linalg.LinAlgError as e:
    print("Cholesky failed:", e)

**Your prediction** *(write it before running the next cell)*:

In [ ]:
def gauss_sample_psd(mu, Sigma, N, rng):
    # YOUR CODE HERE
    raise NotImplementedError

Y = gauss_sample_psd(np.zeros(p), S20, 5000, np.random.default_rng(4))
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(d)** What is the cost of one sample after the factorization, and of the factorization itself, in flops? Which dominates if you need $N = p$ samples?

**Your answer.** *Write it here.*

---
# Extensions — attempt **at most one** in the whole lab

End your extension with a paragraph headed **Finding**: one claim you can defend at the defence, supported by a number in this notebook.

## E1. Repairing a correlation matrix that is not PSD

*This topic was not lectured; the introduction below is all you need.*

Correlation matrices estimated from data with **missing values** are often computed *pairwise*: entry $(i,j)$ uses only the rows where both features $i$ and $j$ were observed. Each entry is then a valid correlation, but the matrix as a whole need not be PSD, because different entries come from different rows. A non-PSD "correlation" matrix breaks Cholesky, sampling and portfolio optimisation, so it must be repaired. The natural repair is the **nearest** correlation matrix:

$$\min\ \|C - R\|_F \quad\text{over symmetric } C\succeq 0 \text{ with } c_{ii}=1.$$

The feasible set is the intersection of two closed convex sets: the PSD cone, and the affine set of symmetric matrices with unit diagonal. Projection onto each set separately is easy; projection onto the intersection is not.

**(a)** Delete 10% of the entries of the breast cancer data at random (seed below), and compute the pairwise correlation matrix with `pandas.DataFrame.corr()`. Show that it is indefinite.

In [ ]:
rng = np.random.default_rng(0)
Xmiss = X.copy()
Xmiss[rng.random(X.shape) < 0.1] = np.nan
Rpw = pd.DataFrame(Xmiss).corr().values
print("most negative eigenvalue of the pairwise correlation:", np.linalg.eigvalsh(Rpw)[0])

**(b)** *Prove:* if $R = U\operatorname{diag}(\lambda)U^\top$, then the PSD matrix nearest to $R$ in the Frobenius norm is $U\operatorname{diag}(\max(\lambda,0))U^\top$.

*Hint.* The Frobenius norm is orthogonally invariant, so put $B = U^\top C U$ and compare $\|B - \operatorname{diag}(\lambda)\|_F^2$ with its diagonal part. What do you know about the diagonal of a PSD matrix?

**Your answer.** *Write it here.*

**(c)** Implement `proj_psd(A)` from (b) and `clip_rescale(R)`, which clips and then rescales to unit diagonal, $C \mapsto D^{-1/2}CD^{-1/2}$ with $D = \operatorname{diag}(C)$. Show numerically that the result is a correlation matrix but is **not** the nearest one. (You will have the nearest one in (d) to compare with.)

**(d)** Implement Higham's alternating projections (N. J. Higham, *Computing the nearest correlation matrix — a problem from finance*, IMA J. Numer. Anal. 22, 2002). Repeat:

$$R_k = Y_{k-1} - \Delta S_{k-1},\quad X_k = P_{\mathrm{PSD}}(R_k),\quad \Delta S_k = X_k - R_k,\quad Y_k = P_{\mathrm{unit}}(X_k),$$

starting from $Y_0 = R$, $\Delta S_0 = 0$. Here $P_{\mathrm{unit}}$ sets the diagonal to 1, and $\Delta S$ is **Dykstra's correction**. Stop when $\|Y_k - Y_{k-1}\|_F$ is below a tolerance.
- Compare the Frobenius distance to $R$ of the result of (c) and of Higham's method.
- Then omit Dykstra's correction (always $\Delta S = 0$) and run again. The iteration still returns a correlation matrix. Is it the nearest one? Compare the distances, and explain the geometric reason in terms of projections onto convex sets.

In [ ]:
def proj_psd(A):
    # YOUR CODE HERE
    raise NotImplementedError

def clip_rescale(R):
    # YOUR CODE HERE
    raise NotImplementedError

def nearest_corr(R, dykstra=True, tol=1e-12, maxit=10000):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def report(name, C):
    print(f"{name:22s} distance {np.linalg.norm(C - Rpw):.5f}, unit diagonal {np.allclose(np.diag(C), 1)}, "
          f"lambda_min {np.linalg.eigvalsh(C)[0]: .1e}")

report("clip and rescale", clip_rescale(Rpw))
C_h, its = nearest_corr(Rpw)
report(f"Higham ({its} iterations)", C_h)
C_nd, its_nd = nearest_corr(Rpw, dykstra=False)
report(f"no Dykstra ({its_nd} it.)", C_nd)
print("lower bound from (b), distance to the nearest PSD matrix:", np.linalg.norm(proj_psd(Rpw) - Rpw))

**Your answer.** *Write it here.*

## E2. Fisher's discriminant as a generalised eigenproblem

*Data:* `sklearn.datasets.load_wine`, 178 wines from 3 cultivars, 13 standardised chemical features.

Fisher's linear discriminant looks for directions $\mathbf{w}$ maximising $J(\mathbf{w}) = \dfrac{\mathbf{w}^\top S_B\mathbf{w}}{\mathbf{w}^\top S_W\mathbf{w}}$, where $S_W$ is the within-class scatter and $S_B$ the between-class scatter. Stationary points satisfy $S_B\mathbf{w} = \lambda S_W\mathbf{w}$ (frame `gen-eig`).

**(a)** Implement `scatter_matrices(X, y)`, returning $S_W = \frac1n\sum_k\sum_{i\in k}(\mathbf{x}_i-\bar{\mathbf{x}}_k)(\mathbf{x}_i-\bar{\mathbf{x}}_k)^\top$ and $S_B = \frac1n\sum_k n_k(\bar{\mathbf{x}}_k-\bar{\mathbf{x}})(\bar{\mathbf{x}}_k-\bar{\mathbf{x}})^\top$. *Before computing, predict* the rank of $S_B$; then confirm it.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
Xw, yw = load_wine(return_X_y=True)
Xw = (Xw - Xw.mean(axis=0)) / Xw.std(axis=0)

def scatter_matrices(X, y):
    # YOUR CODE HERE
    raise NotImplementedError

SW, SB = scatter_matrices(Xw, yw)
print("rank S_B =", np.linalg.matrix_rank(SB), "; cond S_W =", round(np.linalg.cond(SW), 1))

**Your answer.** *Write it here.*

**(b)** Solve $S_B\mathbf{w} = \lambda S_W\mathbf{w}$ in two ways: with `scipy.linalg.eigh(SB, SW)`, and with `np.linalg.eig(np.linalg.solve(SW, SB))`.
- Compare the eigenvalues.
- Check which set of eigenvectors satisfies $V^\top S_WV = I$.
- Explain why the first route returns real eigenvalues by construction and the second does not.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(c)** Project the data onto the two leading directions from `eigh` and plot it, coloured by cultivar.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**(d)** Append a 14th feature that is an almost exact copy of the first, `Xw[:, 0] + eps * noise`, for `eps` $= 10^{-2}, 10^{-4}, \dots, 10^{-10}$ (noise fixed below). Track $\operatorname{cond}(S_W)$ and the two leading eigenvalues from both routes. At what `eps` do the routes part company, and which one do you trust? What would you do in practice?

**Your prediction** *(write it before running the next cell)*:

In [ ]:
noise = np.random.default_rng(5).standard_normal(len(Xw))
for eps in [1e-2, 1e-4, 1e-6, 1e-8, 1e-10]:
    Xa = np.column_stack([Xw, Xw[:, 0] + eps * noise])
    SWa, SBa = scatter_matrices(Xa, yw)
    # YOUR CODE HERE
    raise NotImplementedError
    print(f"eps {eps:.0e}: cond(S_W) {np.linalg.cond(SWa):.1e} | eigh {top_eigh} | eig {top_eig}")

**Your answer.** *Write it here.*

---
**Before you submit:** run `Kernel → Restart & Run All`, check that every written answer is filled in, and make sure your extension (if any) ends with a **Finding**.